# پاکسازی داده ها

In [ ]:
import pandas as pd

# ---------- 1) خواندن فایل CSV ----------
df = pd.read_csv("/content/all_years_movies.csv")

# ---------- 2) پاک‌سازی اولیه ----------
missing_pct = df.isna().mean() * 100
cols_to_drop = missing_pct[missing_pct > 50].index.tolist()
df_clean = df.drop(columns=cols_to_drop)

for col in df_clean.columns:
    if df_clean[col].dtype == object:
        df_clean[col] = df_clean[col].str.strip()

def to_numeric(series):
    return pd.to_numeric(series.str.replace(r"[٬, ]", "", regex=True), errors="ignore")

for num in ["rating", "viewers", "total_sales"]:
    if num in df_clean.columns and df_clean[num].dtype == object:
        df_clean[num] = to_numeric(df_clean[num])

df_clean = df_clean.drop_duplicates()

# ---------- 3) وان-هات‌انکود کردن ژانر ----------
if "genres" in df_clean.columns:
    # 3-الف) استخراج تمام ژانرهای منحصربه‌فرد
    get_set = lambda s: {g.strip() for g in str(s).split(",") if g.strip()}
    unique_genres = sorted({g for genres in df_clean["genres"].dropna() for g in get_set(genres)})

    # 3-ب) ایجاد ستون‌های وان-هات
    for g in unique_genres:
        df_clean[f"genre_{g}"] = df_clean["genres"].apply(lambda x: 1 if g in get_set(x) else 0)




#یافتن داده پرت

In [ ]:

df_clean.describe()

#حذف داده پرت

In [ ]:
df_clean = df_clean[df_clean["rating"] != 10]

# ---------- 4) ذخیرهٔ خروجی ----------
df_clean.to_csv("all_years_movies_cleaned_ohe.csv", index=False, encoding="utf-8-sig")
print("✅ پاک‌سازی و وان-هات‌انکود ژانر انجام شد. فایل «all_years_movies_cleaned_ohe.csv» ذخیره شد.")
